# Chart Reasoning Evaluation

Tests the **reasoning** ability of the LLM on every chart folder in this project (`Version_1` ... `Version_4`): temporal alignment ("when X peaked, what was price doing?"), cross-indicator agreement, and cross-image synthesis for the split versions.

- Each folder has its **own tailored question list** (`QUESTIONS_BY_VERSION`), matched to the indicators that version displays. A folder without a tailored list falls back to `DEFAULT_QUESTIONS`, so new folders still work.
- Answers are **constrained** (the model must pick one of the allowed answers) plus a mandatory short **reasoning** citing the visual evidence and the image(s) used.
- All images of a folder are sent in **one request** (labeled `Image 1`, `Image 2`, ...) so the model can reason across images.
- No ground truth, no scoring: each answer is printed and everything is saved to `llm_reasoning_by_version.json` for manual evaluation.


In [ ]:
from openai import OpenAI

# ------------------------------
# CONFIG
# ------------------------------
API_KEY = ""  # <-- paste your key here
BASE_URL = "https://finderllm2-resource.services.ai.azure.com/openai/v1"
MODEL = "gpt-5-mini"
ROOT_FOLDER = "."
OUTPUT_FILE = "llm_reasoning_by_version.json"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)


In [ ]:
# ------------------------------
# QUESTION SETS - tailored per chart version
# Each question: id, panel, question, allowed_answers (no expected answers - evaluation is manual).
# Questions are phrased relative to the visible window ("at the end", "sharpest decline"),
# because every version covers a different date range.
# ------------------------------

QUESTIONS_BY_VERSION = {

    # 1 image, 6 panels: Close+BB Lower+BB Width | Volatility+ATR | MACD+RSI | MaxDD+Ulcer | VaR/ES | OBV
    "Version_1": [
        {"id": 1, "panel": "Price & Trend",
         "question": "Over the final month visible, is the Close price in an uptrend, a downtrend, or moving sideways?",
         "allowed_answers": ["Uptrend", "Downtrend", "Sideways"]},
        {"id": 2, "panel": "Price & Trend",
         "question": "Does the Close price touch or fall below the BB Lower line at any point during the last two months visible?",
         "allowed_answers": ["Yes", "No"]},
        {"id": 3, "panel": "Price & Trend",
         "question": "In the period when BB Width (right axis) reaches its maximum, is the Close price generally rising or falling?",
         "allowed_answers": ["Rising", "Falling", "Flat"]},
        {"id": 4, "panel": "Volatility & ATR",
         "question": "In which month does ATR reach its highest value?",
         "allowed_answers": ["2024-07", "2024-08", "2024-09", "2024-10", "2024-11", "2024-12"]},
        {"id": 5, "panel": "Volatility & ATR + Price",
         "question": "Around the time ATR reaches its maximum, is the Close price rising, falling, or flat?",
         "allowed_answers": ["Rising", "Falling", "Flat"]},
        {"id": 6, "panel": "MACD & RSI",
         "question": "At the last visible data point, is RSI above the overbought line (70), below the oversold line (30), or between them?",
         "allowed_answers": ["Above 70", "Below 30", "Between"]},
        {"id": 7, "panel": "MACD & RSI",
         "question": "At the last visible data point, is the MACD histogram positive or negative?",
         "allowed_answers": ["Positive", "Negative"]},
        {"id": 8, "panel": "MACD & RSI",
         "question": "Considering RSI and the MACD histogram together at the end of the chart, do they give the same directional signal?",
         "allowed_answers": ["Both bullish", "Both bearish", "Mixed"]},
        {"id": 9, "panel": "Drawdown Risk",
         "question": "In which month does the Ulcer Index reach its maximum?",
         "allowed_answers": ["2024-07", "2024-08", "2024-09", "2024-10", "2024-11", "2024-12"]},
        {"id": 10, "panel": "Drawdown Risk + VaR/ES",
         "question": "Are the VaR and ES lines at their most negative during the same period in which the Ulcer Index peaks?",
         "allowed_answers": ["Yes", "No"]},
        {"id": 11, "panel": "OBV + Price",
         "question": "Over the final month visible, does OBV move in the same direction as the Close price (confirmation) or in the opposite direction (divergence)?",
         "allowed_answers": ["Confirmation", "Divergence"]},
        {"id": 12, "panel": "Synthesis",
         "question": "Combining the momentum panel (MACD/RSI) and the risk panels (Ulcer Index, VaR/ES) at the last visible data point, is the overall picture risk-on (calm/bullish), risk-off (stressed/bearish), or mixed?",
         "allowed_answers": ["Risk-on", "Risk-off", "Mixed"]},
    ],

    # 1 tall image, ~11 dense panels, early-2020 window with a large crash
    "Version_2": [
        {"id": 1, "panel": "Price & Trend",
         "question": "At the last visible data point, is the Close price above or below SMA200?",
         "allowed_answers": ["Above", "Below"]},
        {"id": 2, "panel": "Price & Trend",
         "question": "In which month does the sharpest price decline occur?",
         "allowed_answers": ["2020-01", "2020-02", "2020-03", "2020-04", "2020-05", "2020-06"]},
        {"id": 3, "panel": "Oscillators + Price",
         "question": "During the sharpest price decline, does RSI fall below the oversold line (30)?",
         "allowed_answers": ["Yes", "No"]},
        {"id": 4, "panel": "Volatility",
         "question": "At the last visible data point, is HV_30 near its maximum for the window, near its minimum, or somewhere in the middle?",
         "allowed_answers": ["Near maximum", "Near minimum", "Middle"]},
        {"id": 5, "panel": "Oscillators",
         "question": "At the last visible data point, is the MACD histogram positive or negative?",
         "allowed_answers": ["Positive", "Negative"]},
        {"id": 6, "panel": "Stoch / WilliamsR / Z-Score",
         "question": "At the last visible data point, are Stoch %K and Stoch %D both above 80?",
         "allowed_answers": ["Yes", "No"]},
        {"id": 7, "panel": "ATR / CMF",
         "question": "At the last visible data point, is CMF above or below zero?",
         "allowed_answers": ["Above", "Below"]},
        {"id": 8, "panel": "OBV / ADX",
         "question": "Over the final two months visible, is OBV rising or falling?",
         "allowed_answers": ["Rising", "Falling"]},
        {"id": 9, "panel": "OBV + Price",
         "question": "Does the OBV direction over the final two months confirm the Close price direction over the same period?",
         "allowed_answers": ["Confirms", "Diverges"]},
        {"id": 10, "panel": "Skewness & Kurtosis",
         "question": "At the last visible data point, is the 20-day skewness positive or negative?",
         "allowed_answers": ["Positive", "Negative"]},
        {"id": 11, "panel": "VaR & ES + Price",
         "question": "At the last visible data point, are the VaR/ES lines wider (more negative) or narrower (closer to zero) than during the sharpest-decline period?",
         "allowed_answers": ["Wider", "Narrower", "About the same"]},
        {"id": 12, "panel": "Synthesis",
         "question": "Considering price versus its moving averages, volatility, and the risk panels at the end of the chart, does the market look like it is recovering from the earlier decline or still under stress?",
         "allowed_answers": ["Recovering", "Still under stress", "Mixed"]},
    ],

    # 3 images: 1 = Price/Volatility/Oscillators/ROC, 2 = ATR+Volume/CMF/Stoch/Ulcer+CCI, 3 = OBV+ADX/Skew+Kurt/VaR+ES
    # Several questions force cross-image reasoning.
    "Version_3": [
        {"id": 1, "panel": "Image 1 - Price & Trend",
         "question": "At the last visible data point, is the Close price above or below SMA200?",
         "allowed_answers": ["Above", "Below"]},
        {"id": 2, "panel": "Image 1 - Price & Trend",
         "question": "The price panel marks a Death Cross event. In which month does it occur?",
         "allowed_answers": ["2024-11", "2024-12", "2025-01", "2025-02", "2025-03", "2025-04", "2025-05"]},
        {"id": 3, "panel": "Image 1 - Price & Trend",
         "question": "In which month does the sharpest price decline occur?",
         "allowed_answers": ["2024-11", "2024-12", "2025-01", "2025-02", "2025-03", "2025-04", "2025-05"]},
        {"id": 4, "panel": "Cross-image: Image 1 + Image 2",
         "question": "During the sharpest price decline (visible in Image 1), is ATR (Image 2) rising or falling?",
         "allowed_answers": ["Rising", "Falling"]},
        {"id": 5, "panel": "Image 1 - Oscillators",
         "question": "At the last visible data point, is RSI above the overbought line (70), below the oversold line (30), or between them?",
         "allowed_answers": ["Above 70", "Below 30", "Between"]},
        {"id": 6, "panel": "Image 1 - Oscillators",
         "question": "At the last visible data point, is the MACD histogram positive or negative?",
         "allowed_answers": ["Positive", "Negative"]},
        {"id": 7, "panel": "Image 2 - Stochastic",
         "question": "At the last visible data point, are Stoch %K and Stoch %D both above 80?",
         "allowed_answers": ["Yes", "No"]},
        {"id": 8, "panel": "Image 2 - CMF",
         "question": "At the last visible data point, is CMF above or below zero?",
         "allowed_answers": ["Above", "Below"]},
        {"id": 9, "panel": "Image 3 - ADX",
         "question": "At the last visible data point, is ADX above or below the strong-trend threshold (25)?",
         "allowed_answers": ["Above", "Below"]},
        {"id": 10, "panel": "Cross-image: Image 3 + Image 1",
         "question": "Over the final month visible, does OBV (Image 3) confirm the Close price direction (Image 1) or diverge from it?",
         "allowed_answers": ["Confirms", "Diverges"]},
        {"id": 11, "panel": "Cross-image: Image 3 + Image 1",
         "question": "Do the VaR/ES lines (Image 3) reach their most negative values during the same period as the sharpest price decline (Image 1)?",
         "allowed_answers": ["Yes", "No"]},
        {"id": 12, "panel": "Synthesis - all images",
         "question": "Combining trend and momentum (Image 1), volatility and money flow (Image 2), and the risk measures (Image 3) at the last visible data point, is the overall picture risk-on, risk-off, or mixed?",
         "allowed_answers": ["Risk-on", "Risk-off", "Mixed"]},
    ],

    # 6 sparse images: 1 = Close+SMA200 / VOL_GK+Vol_1pct, 2 = RSI+MACD Hist / Ret_99pct,
    # 3 = ATR+Volume / WilliamsR+Z-Score, 4 = Ulcer / OBV, 5 = Skew60+Kurt60, 6 = 5% VaR + 5% ES
    # Cross-image reasoning is unavoidable here.
    "Version_4": [
        {"id": 1, "panel": "Image 1 - Price & Trend",
         "question": "At the last visible data point, is the Close price above or below SMA200?",
         "allowed_answers": ["Above", "Below"]},
        {"id": 2, "panel": "Image 1 - Price & Trend",
         "question": "Over the final month visible, is the Close price in an uptrend, a downtrend, or moving sideways?",
         "allowed_answers": ["Uptrend", "Downtrend", "Sideways"]},
        {"id": 3, "panel": "Image 1 - Volatility",
         "question": "At the last visible data point, which line is higher: VOL_GK or Vol_1pct?",
         "allowed_answers": ["VOL_GK", "Vol_1pct"]},
        {"id": 4, "panel": "Image 2 - Oscillators",
         "question": "At the last visible data point, is RSI above the overbought line (70), below the oversold line (30), or between them?",
         "allowed_answers": ["Above 70", "Below 30", "Between"]},
        {"id": 5, "panel": "Image 2 - Oscillators",
         "question": "At the last visible data point, is the MACD histogram positive or negative?",
         "allowed_answers": ["Positive", "Negative"]},
        {"id": 6, "panel": "Image 2 - Oscillators",
         "question": "Considering RSI and the MACD histogram together at the end of the chart, do they give the same directional signal?",
         "allowed_answers": ["Both bullish", "Both bearish", "Mixed"]},
        {"id": 7, "panel": "Image 3 - ATR",
         "question": "Over the final month visible, is ATR rising or falling?",
         "allowed_answers": ["Rising", "Falling"]},
        {"id": 8, "panel": "Image 3 - Williams %R",
         "question": "At the last visible data point, is Williams %R near 0 (top of its range), near -100 (bottom of its range), or in the middle?",
         "allowed_answers": ["Near 0", "Near -100", "Middle"]},
        {"id": 9, "panel": "Image 4 - Ulcer Index",
         "question": "Over the final month visible, is the Ulcer Index rising or falling, and is its final value near the maximum of the window?",
         "allowed_answers": ["Rising, near maximum", "Rising, not near maximum", "Falling, near maximum", "Falling, not near maximum"]},
        {"id": 10, "panel": "Cross-image: Image 4 + Image 1",
         "question": "Over the final month visible, does OBV (Image 4) confirm the Close price direction (Image 1) or diverge from it?",
         "allowed_answers": ["Confirms", "Diverges"]},
        {"id": 11, "panel": "Image 5 - Skewness",
         "question": "At the last visible data point, is the 60-day skewness positive or negative?",
         "allowed_answers": ["Positive", "Negative"]},
        {"id": 12, "panel": "Image 6 - VaR & ES",
         "question": "At the last visible data point, is the 5% ES line below (more negative than) the 5% VaR line?",
         "allowed_answers": ["Yes", "No"]},
        {"id": 13, "panel": "Synthesis - all images",
         "question": "Combining all six images at the last visible data point, is overall market risk rising, falling, or mixed? In your reasoning, name the two images that most support your answer.",
         "allowed_answers": ["Rising", "Falling", "Mixed"]},
    ],
}

# Fallback for any folder without a tailored list (indicator-generic, "Not visible" allowed).
DEFAULT_QUESTIONS = [
    {"id": 1, "panel": "Price",
     "question": "Over the final month visible, is the main price series in an uptrend, a downtrend, or moving sideways?",
     "allowed_answers": ["Uptrend", "Downtrend", "Sideways"]},
    {"id": 2, "panel": "Price",
     "question": "In which part of the window does the sharpest price decline occur?",
     "allowed_answers": ["Beginning", "Middle", "End", "No clear decline"]},
    {"id": 3, "panel": "Volatility",
     "question": "If a volatility measure is shown, is it rising or falling over the final month visible?",
     "allowed_answers": ["Rising", "Falling", "Not visible"]},
    {"id": 4, "panel": "Oscillators",
     "question": "If RSI is shown, at the last visible data point is it above 70, below 30, or between them?",
     "allowed_answers": ["Above 70", "Below 30", "Between", "Not visible"]},
    {"id": 5, "panel": "Oscillators",
     "question": "If the MACD histogram is shown, is it positive or negative at the last visible data point?",
     "allowed_answers": ["Positive", "Negative", "Not visible"]},
    {"id": 6, "panel": "Volume / Flow",
     "question": "If a volume or money-flow indicator (OBV, CMF) is shown, does it confirm the price direction over the final month?",
     "allowed_answers": ["Confirms", "Diverges", "Not visible"]},
    {"id": 7, "panel": "Risk",
     "question": "If risk measures (VaR, ES, Ulcer Index) are shown, are they more extreme at the end of the window than at the beginning?",
     "allowed_answers": ["More extreme", "Less extreme", "About the same", "Not visible"]},
    {"id": 8, "panel": "Synthesis",
     "question": "Overall, at the last visible data point, is the picture risk-on, risk-off, or mixed?",
     "allowed_answers": ["Risk-on", "Risk-off", "Mixed"]},
]


In [ ]:
import os
import base64
import json

# ------------------------------
# HELPERS
# ------------------------------
VALID_EXT = (".png", ".jpg", ".jpeg", ".webp")

def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

def guess_mime_type(path):
    p = path.lower()
    if p.endswith(".png"):
        return "image/png"
    if p.endswith(".jpg") or p.endswith(".jpeg"):
        return "image/jpeg"
    if p.endswith(".webp"):
        return "image/webp"
    return "application/octet-stream"

def get_images(folder):
    return sorted(
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(VALID_EXT)
    )

def get_chart_folders(root):
    """Every subfolder of root that contains at least one image (dot-folders skipped)."""
    return [
        os.path.join(root, d)
        for d in sorted(os.listdir(root))
        if not d.startswith(".")
        and os.path.isdir(os.path.join(root, d))
        and get_images(os.path.join(root, d))
    ]

def strip_fences(raw):
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        parts = cleaned.split("```")
        if len(parts) >= 2:
            cleaned = parts[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
    return cleaned.strip()

# ------------------------------
# PROMPT
# ------------------------------
PROMPT_TEMPLATE = """
You are analyzing {N_IMAGES} financial chart image(s) that belong to ONE chart set.
The images are labeled Image 1 .. Image {N_IMAGES}, in the order they appear below.

Answer the questions using ONLY the visual information in the images.

Rules:
- "model_answer" must be EXACTLY one of the allowed_answers of that question.
- "reasoning" must be 2-3 short sentences describing the visual evidence you used.
- "evidence_images" must list the image numbers you used (e.g. [1] or [1, 3]).
- If a question says "at the end" or "at the last visible data point", focus on the rightmost plotted values.
- Do not use outside financial knowledge; rely only on what is visible in the images.

Questions:
{QUESTIONS_JSON}

Return ONLY valid JSON in exactly this format:

{
  "answers": [
    {
      "id": 1,
      "model_answer": "...",
      "reasoning": "...",
      "evidence_images": [1]
    }
  ]
}

- Give exactly one answer object per question.
- Do not include markdown fences or any text outside the JSON.
"""

def build_prompt(n_images, questions):
    return (
        PROMPT_TEMPLATE
        .replace("{N_IMAGES}", str(n_images))
        .replace("{QUESTIONS_JSON}", json.dumps(questions, indent=2))
    )

def build_messages(prompt_text, image_paths):
    content = [{"type": "text", "text": prompt_text}]
    for i, path in enumerate(image_paths, start=1):
        content.append({"type": "text", "text": f"Image {i}: {os.path.basename(path)}"})
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:{guess_mime_type(path)};base64,{encode_image(path)}"},
        })
    return [{"role": "user", "content": content}]

# ------------------------------
# STRUCTURED OUTPUT + FALLBACK CHAIN
# ------------------------------
RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "chart_reasoning_eval",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "answers": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id": {"type": "integer"},
                            "model_answer": {"type": "string"},
                            "reasoning": {"type": "string"},
                            "evidence_images": {"type": "array", "items": {"type": "integer"}},
                        },
                        "required": ["id", "model_answer", "reasoning", "evidence_images"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["answers"],
            "additionalProperties": False,
        },
    },
}

def call_model(messages):
    """json_schema -> json_object -> plain prompt. Returns (parsed_or_raw, call_mode)."""
    try:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, response_format=RESPONSE_SCHEMA
        )
        return json.loads(response.choices[0].message.content), "json_schema"
    except Exception:
        pass
    try:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, response_format={"type": "json_object"}
        )
        return json.loads(strip_fences(response.choices[0].message.content)), "json_object"
    except Exception:
        pass
    response = client.chat.completions.create(model=MODEL, messages=messages)
    raw = strip_fences(response.choices[0].message.content)
    try:
        return json.loads(raw), "plain_prompted_json"
    except json.JSONDecodeError:
        return raw, "unparsed_text"

chart_folders = get_chart_folders(ROOT_FOLDER)
print(f"Found {len(chart_folders)} chart folder(s):")
for folder in chart_folders:
    name = os.path.basename(folder)
    tailored = "tailored" if name in QUESTIONS_BY_VERSION else "default"
    n_q = len(QUESTIONS_BY_VERSION.get(name, DEFAULT_QUESTIONS))
    print(f"  {name}: {len(get_images(folder))} image(s), {n_q} {tailored} question(s)")


In [ ]:
# ------------------------------
# PROCESS ALL CHART FOLDERS
# ------------------------------
results = {}

for chart_folder in chart_folders:
    chart_name = os.path.basename(chart_folder)
    images = get_images(chart_folder)
    questions = QUESTIONS_BY_VERSION.get(chart_name, DEFAULT_QUESTIONS)

    print(f"\n{'=' * 80}")
    print(f"\U0001F4C1 {chart_name} - {len(images)} image(s), {len(questions)} question(s)")
    print("=" * 80)

    prompt_text = build_prompt(len(images), questions)
    messages = build_messages(prompt_text, images)
    parsed, call_mode = call_model(messages)
    print(f"(call mode: {call_mode})\n")

    if isinstance(parsed, dict):
        q_by_id = {q["id"]: q for q in questions}
        for ans in parsed.get("answers", []):
            q = q_by_id.get(ans.get("id"), {})
            print(f"Q{ans.get('id')} [{q.get('panel', '?')}] {q.get('question', '')}")
            print(f"   \u27A4 {ans.get('model_answer')}")
            print(f"   reasoning: {ans.get('reasoning')}")
            print(f"   evidence:  Image(s) {ans.get('evidence_images')}\n")
    else:
        # JSON could not be parsed - keep and show the raw text so nothing is lost
        print(parsed)

    results[chart_name] = {
        "images": [os.path.basename(p) for p in images],
        "call_mode": call_mode,
        "questions": questions,
        "answers": parsed,
    }

with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n\u2705 All outputs saved to {OUTPUT_FILE}")
